# 02 — Day-ahead load forecasting: baselines (SeasonalNaive, LightGBM, ONS)

**Task.** Forecast issued at **D-1 12:00** with data up to 11:00, for the 24 hours of D (steps 13–36),
the same information set as the **ONS programmed load**, which is the external benchmark.
`ONS_adj` is the programmed load rescaled to the curva-carga level with a ratio known at D-1 (causal).

**Backtest.** Rolling origin, one cutoff per day; train from 2015, test from 2024-01-01 to the last
full day (includes the 2026 World Cup). LightGBM refits every 30 days. Point models get causal
rolling conformal intervals.

> ⚠️ This notebook uses **LightGBM** and must not import torch (macOS libomp clash). The deep models
> are in `03_forecast_deep.ipynb`; the final comparison there reads the backtests stored here.

Results are cached in `data/interim/forecasts/<name>.parquet`; set `RETRAIN = True` to refit.
Runtime on an M4: ~10–15 min for SE/CO.

In [ ]:
%matplotlib inline
import pandas as pd

from ons_energy.forecast.backtest import last_full_day, mase_scales, run_or_load
from ons_energy.forecast.config import BacktestConfig, ModelSpec
from ons_energy.forecast.evaluate import attach_day_types, dm_table, evaluate_backtest
from ons_energy.tracking import log_backtest
from ons_energy.viz import forecast_plots as F

pd.set_option("display.precision", 3)
RETRAIN = False  # True: refit even if a stored backtest exists
LOG_MLFLOW = True
RUN_WEATHER_ABLATION = False  # global LightGBM on 4 subsystems x 3 weather scenarios (~1 h)

## Experiment

In [ ]:
NAIVE = ModelSpec(name="SeasonalNaive", kind="seasonal_naive", params={"season_length": 168})
LGBM = ModelSpec(name="LightGBM", kind="lgbm")
ONS = ModelSpec(name="ONS", kind="ons_programmed")

cfg = BacktestConfig(
    name="mvp_seco_baselines",
    subsystems=["SE"],
    weather="none",
    refit_every_days=30,
    models=[NAIVE, LGBM, ONS],
)
cfg

In [ ]:
cv = run_or_load(cfg, retrain=RETRAIN)
tabs = evaluate_backtest(cv, mase_scales(cfg), reference="ONS")
if LOG_MLFLOW:
    log_backtest(cfg, tabs, cv)
print(f"{cv.target_date.min()} -> {cv.target_date.max()}, {cv.target_date.nunique()} days")
tabs["overall"].sort_values("MAPE")

## Actual vs predicted
Whole test period (daily means and 7-day rolling MAPE), then hourly zooms on a normal week and on
special days.

In [ ]:
MODELS = ["SeasonalNaive", "LightGBM", "ONS_adj"]
fig = F.actual_vs_forecast_daily(cv, ["LightGBM", "ONS_adj"])

In [ ]:
fig = F.actual_vs_forecast(cv, MODELS, "2024-03-11", band_model="LightGBM", title="SE/CO — a normal week (LightGBM intervals shaded)")

In [ ]:
fig = F.actual_vs_forecast(cv, MODELS, "2025-02-27", band_model="LightGBM", title="SE/CO — Carnival 2025 (Mon 03/03, Tue 04/03)")

In [ ]:
fig = F.actual_vs_forecast(cv, MODELS, "2024-11-13", days=9, band_model="LightGBM",
                           title="SE/CO — national holidays 15/11 (Fri) and 20/11 (Wed), 2024")

In [ ]:
fig = F.actual_vs_forecast(cv, MODELS, "2024-12-21", days=14, title="SE/CO — Christmas and New Year 2024/25")

In [ ]:
fig = F.actual_vs_forecast(cv, MODELS, "2026-06-12", days=8, band_model="LightGBM",
                           title="SE/CO — World Cup 2026: BRA×MAR 13/06 19:00, BRA×HAI 19/06 22:00")

In [ ]:
fig = F.actual_vs_forecast(cv, MODELS, "2026-06-23", days=8, band_model="LightGBM",
                           title="SE/CO — World Cup 2026: BRA×SCO 24/06 19:00, BRA×JPN 29/06 14:00")

## Errors by day type and by hour
Day-bootstrap 95% CIs; `n` = number of days in each segment.

In [ ]:
seg = tabs["by_day_type"]
seg.pivot_table(index="day_type", columns="model", values="MAPE", observed=True).assign(
    n_days=seg.groupby("day_type", observed=True)["n_days"].max()).round(2)

In [ ]:
fig = F.mape_by_day_type(seg, MODELS, "SE/CO MAPE by day type (test period)")

In [ ]:
fig = F.mape_by_hour(tabs["by_step"], MODELS, "SE/CO MAPE by hour of the target day")

## Diebold-Mariano tests (daily MAPE; negative = better than the reference)

In [ ]:
tabs["dm"].sort_values("DM")  # vs ONS programmed load

In [ ]:
dm_table(attach_day_types(cv), "SeasonalNaive").sort_values("DM")

## Interpretability: LightGBM SHAP (normal vs special days)
Exact TreeSHAP (`pred_contrib=True`) of the one-step model trained up to the test start.

In [ ]:
from ons_energy.forecast.interpret import lgbm_shap

shap = lgbm_shap(cfg, cfg.eval_end(last_full_day()))["lgbm_shap"]
top = shap.pivot_table(index="feature", columns="segment", values="mean_abs_shap").sort_values("special", ascending=False).head(15)
ax = top[::-1].plot.barh(figsize=(8, 5), title="LightGBM mean |SHAP| (MW), top 15 features")
top.round(1)

## Optional: global LightGBM on 4 subsystems and weather scenarios
`none` = no weather; `perfect` = reanalysis used as if it were a forecast (upper bound);
`honest` = forecast issued the day before (Open-Meteo previous runs, from 2022).

In [ ]:
if RUN_WEATHER_ABLATION:
    SUBS = ["SE", "S", "NE", "N"]
    scenarios = {
        "none": BacktestConfig(name="global_baselines_none", subsystems=SUBS, models=[NAIVE, LGBM, ONS]),
        "perfect": BacktestConfig(name="global_lgbm_perfect", subsystems=SUBS, weather="perfect", models=[LGBM]),
        "honest": BacktestConfig(name="global_lgbm_honest", subsystems=SUBS, weather="honest", models=[LGBM]),
    }
    by_sub = {}
    for scen, c in scenarios.items():
        cv_w = run_or_load(c, retrain=RETRAIN)
        t = evaluate_backtest(cv_w, mase_scales(c), reference="ONS" if "ONS" in cv_w else None)
        if LOG_MLFLOW:
            log_backtest(c, t, cv_w)
        by_sub[scen] = t["by_subsystem"]
    fig = F.weather_ablation(by_sub)
    display(pd.concat({k: v.pivot_table(index="model", columns="unique_id", values="MAPE") for k, v in by_sub.items()}).round(2))